# Does the schema checker measure the agent, or the file format?

Before any violation rate is reported on ToolBench trajectories, the checker has to be
shown to be reading agent behaviour rather than artefacts of how the corpus was
serialised. Three artefacts are plausible: tool names carrying a service prefix that
the schema does not use, the `Finish` pseudo-tool being absent from the declared API
list, and `Action Input` arriving as a string rather than an object.

This notebook parses the corpus, applies the paper's two transferable schema checks,
and prints twenty flagged calls in full alongside the schema they were checked
against, so that each one can be judged by hand. It reports parse health too. No rate
printed here should be quoted as a result: this is an instrument check.

In [ ]:
import ast, itertools, json, re
from collections import Counter
from datasets import load_dataset

N_RECORDS = 400
ds = load_dataset("tuandunghcmut/toolbench-v1", split="train", streaming=True)
rows = list(itertools.islice(iter(ds), N_RECORDS))
print("records pulled: %d" % len(rows))
print("fields: %s" % sorted(rows[0].keys()))

def turns(r):
    c = r["conversations"]
    if isinstance(c, str):
        c = json.loads(c)
    return list(zip(c["from"], c["value"]))

print("roles in the first record: %s" % [f for f, _ in turns(rows[0])])

In [ ]:
# ---- the schemas, read out of the system turn
APIS = re.compile(r"you have access to the following APIs:\s*(\[.*)", re.S)

def schemas(r):
    for role, val in turns(r):
        if role != "system":
            continue
        m = APIS.search(val)
        if not m:
            return None
        blob = m.group(1).strip()
        depth, end = 0, None
        for i, ch in enumerate(blob):          # couper sur le crochet fermant
            depth += (ch == "[") - (ch == "]")
            if depth == 0:
                end = i + 1
                break
        try:
            return ast.literal_eval(blob[:end])
        except Exception:
            return None
    return None

ok = sum(1 for r in rows if schemas(r))
print("records whose tool schemas parse: %d of %d" % (ok, len(rows)))
s0 = schemas(rows[0])
print("first record declares %d tools: %s" % (len(s0), [t["name"] for t in s0]))
print("\nis Finish declared as a tool?")
has_finish = sum(1 for r in rows if (schemas(r) or []) and
                 any(t.get("name") == "Finish" for t in schemas(r)))
print("   declared in %d of %d records" % (has_finish, ok))

In [ ]:
# ---- the steps, read out of the assistant turns
STEP = re.compile(r"Action:\s*(?P<action>[^\n]*)\n+Action Input:\s*(?P<inp>.*)", re.S)

def steps(r):
    out, pending = [], None
    for role, val in turns(r):
        if role == "assistant":
            m = STEP.search(val)
            if not m:
                continue
            thought = val.split("Action:")[0].replace("Thought:", "").strip()
            pending = {"thought": thought,
                       "action": m.group("action").strip(),
                       "input_raw": m.group("inp").strip(),
                       "observation": None}
            out.append(pending)
        elif role == "function" and pending is not None:
            pending["observation"] = val
            pending = None
    return out

counts = Counter()
for r in rows:
    st = steps(r)
    counts["records"] += 1
    counts["records with >=1 step"] += bool(st)
    counts["steps"] += len(st)
    counts["steps with observation"] += sum(1 for s in st if s["observation"])
for k, v in counts.items():
    print("%-26s %d" % (k, v))
print("\nmean steps per record: %.2f" % (counts["steps"] / max(1, counts["records"])))

In [ ]:
# ---- how does Action Input arrive, and do the names match the schemas
def parse_input(raw):
    raw = raw.strip()
    for loader in (json.loads, ast.literal_eval):
        try:
            v = loader(raw)
            if isinstance(v, dict):
                return v, "object"
            return v, "non-object:%s" % type(v).__name__
        except Exception:
            pass
    return None, "unparseable"

shape, namefit = Counter(), Counter()
for r in rows:
    names = {t["name"] for t in (schemas(r) or [])}
    for s in steps(r):
        _, how = parse_input(s["input_raw"])
        shape[how] += 1
        a = s["action"]
        if a in names:
            namefit["exact match"] += 1
        elif a == "Finish":
            namefit["Finish, not in schema"] += 1
        elif any(a.startswith(n) or n.startswith(a) for n in names):
            namefit["prefix-related to a declared name"] += 1
        else:
            namefit["no relation to any declared name"] += 1

print("shape of Action Input")
for k, v in shape.most_common():
    print("   %-26s %d" % (k, v))
print("\naction name against the declared tool set")
for k, v in namefit.most_common():
    print("   %-34s %d" % (k, v))

In [ ]:
# ---- the two transferable checks, applied, with every flag kept for inspection
def check(r):
    sc = schemas(r) or []
    by = {t["name"]: t for t in sc}
    flags = []
    for k, s in enumerate(steps(r)):
        a, raw = s["action"], s["input_raw"]
        if a not in by:
            flags.append((k, "tool not in declared set", a, raw, None))
            continue
        spec = by[a].get("parameters", {}) or {}
        req = spec.get("required", []) or []
        val, how = parse_input(raw)
        if how != "object":
            flags.append((k, "input not an object (%s)" % how, a, raw, req))
            continue
        missing = [p for p in req if p not in val]
        if missing:
            flags.append((k, "required missing: %s" % ",".join(missing), a, raw, req))
    return flags

ALL = [(i, f) for i, r in enumerate(rows) for f in check(r)]
kinds = Counter(re.sub(r":.*", "", f[1]) for _, f in ALL)
print("flags raised: %d over %d records" % (len(ALL), len(rows)))
for k, v in kinds.most_common():
    print("   %-34s %d" % (k, v))
print("\nNOTE: these are candidate violations, not results. The point of the packet")
print("below is to decide, by hand, which of them are agent errors at all.")

In [ ]:
# ---- the audit packet: twenty flags in full, spread across the kinds
import random
random.seed(20261018)

bykind = {}
for i, f in ALL:
    bykind.setdefault(re.sub(r":.*", "", f[1]), []).append((i, f))
packet = []
for kind, items in bykind.items():
    random.shuffle(items)
    packet += items[:max(1, 20 // max(1, len(bykind)))]
packet = packet[:20]

for n, (i, (k, why, action, raw, req)) in enumerate(packet, 1):
    sc = schemas(rows[i]) or []
    declared = [t["name"] for t in sc]
    print("=" * 78)
    print("FLAG %d of %d   record %d, step %d" % (n, len(packet), i, k))
    print("reason      : %s" % why)
    print("action       : %r" % action)
    print("action input : %s" % raw[:300])
    print("required     : %s" % req)
    print("declared     : %s" % declared)
    st = steps(rows[i])
    if k < len(st) and st[k]["observation"]:
        print("observation  : %s" % str(st[k]["observation"])[:220])
    print()